In [1]:
import numpy as np
import ifcopenshell

# for some reason ifcopenshell.api is not imported when importing ifcopenshell, so we need to import it explicitly
import ifcopenshell.api.root
import ifcopenshell.api.unit
import ifcopenshell.api.context
import ifcopenshell.api.project
import ifcopenshell.api.geometry

model = ifcopenshell.open("Function_Model_v3.ifc")

# defined as a dictionary, because there can be multiple boundaries related by openings
def get_boundary_room(space):
    b_room = {}
    for boundary in space.BoundedBy:
        corr_boundary = boundary.CorrespondingBoundary
        b_room[boundary.RelatedBuildingElement.Name] = corr_boundary.RelatingSpace
    return b_room

def get_openings(space):
    openings = {}
    for boundary in space.BoundedBy:
        if boundary.RelatedBuildingElement.is_a("IfcOpeningElement"):
            openings[boundary.RelatedBuildingElement.Name] = boundary.RelatedBuildingElement
    return openings

def get_coordinates(entity):
    matrix = ifcopenshell.util.placement.get_local_placement(entity.ObjectPlacement)
    return matrix[:,3][0:3]

###### other functions to add later:
# get length/width of a space
# calculate distance between spaces




Do the edge weightings need to be known before implementing the algorithm?
Probably not, but
since I'm finding the worst case from a given building, there is no "start". So I guess will need to iterate through every path anyway?


In [4]:
import networkx as nx
import plotly.graph_objects as go

def build_nx_space_graph(model):
    """Build a graph of IfcSpaces connected through IfcOpeningElements."""
    G = nx.Graph()

    for space in model.by_type("IfcSpace"):
        s1 = space.Name or f"IfcSpace_{space.id()}"
        G.add_node(s1, ifc_id=space.id())

        for b in getattr(space, "BoundedBy", []):
            if not b.is_a("IfcRelSpaceBoundary2ndLevel"):
                continue

            cb = getattr(b, "CorrespondingBoundary", None)
            if not cb:
                continue

            elem = getattr(b, "RelatedBuildingElement", None)
            if not elem or not elem.is_a("IfcOpeningElement"):
                continue

            s2_obj = cb.RelatingSpace
            s2 = s2_obj.Name or f"IfcSpace_{s2_obj.id()}"
            opening_name = elem.Name or f"IfcOpeningElement_{elem.id()}"

            if not G.has_edge(s1, s2):
                G.add_edge(
                    s1,
                    s2,
                    opening=opening_name,
                    opening_id=elem.id(),
                    boundary_id=b.id(),
                    corresponding_boundary_id=cb.id(),
                )

    return G

G = build_nx_space_graph(model)

In [ ]:

# perhaps just try to make this with one goal first (one exit), then we can try to make it with multiple goals (multiple exits) later.

def h_n(current, goal):
    coords1 = get_coordinates(current)[0:2]
    coords2 = get_coordinates(goal)[0:2]
    distance = np.linalg.norm(coords1 - coords2)
    return distance


# A* algorithm. To be used for multiple starting points and goals, so we can find the shortest path to an exit in the building.
def a_star(start, goal, h):
    open_set = set()
    closed_set = set()

    open_set.add(start)
                          
    g_score = {start: 0}                                    
    f_score = {start: h(start)}                         
    while open_set is not empty: 
        current = node in open_set with lowest f_score 
        if current == goal: 
            return reconstruct_path(current) 
        open_set.remove(current) 
        closed_set.add(current) 
        for neighbor in get_neighbors(current): 
            if neighbor in closed_set: 
                continue 

            # change cost to weights i.e. distance to neighbours i think
            tentative_g = g_score[current] + cost(current, neighbor) 


            if neighbor not in open_set: 
                open_set.add(neighbor) 
            elif tentative_g >= g_score[neighbor]: 
                continue 
            g_score[neighbor] = tentative_g 
            f_score[neighbor] = g_score[neighbor] + h(neighbor)

In [16]:
print(type(G.nodes))

<class 'networkx.classes.reportviews.NodeView'>
